# 🧬 DiffDock ile GPU Doğrulama (Google Colab · ücretsiz T4)

Bu notebook, Remedia pipeline'ında bulduğun aday molekülleri **DiffDock** (derin
öğrenmeli docking) ile ikinci bir yöntemle doğrular. Vina zaten kimyasal skoru
verir; DiffDock bağımsız bir "güven skoru" ekler. İkisi birden güçlüyse molekül
gerçekten umut vaat ediyordur.

### 📋 Nasıl kullanılır — SADECE ŞU 3 KURAL:
1. Üstten **Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ GPU (T4)** seç.
2. Hücreleri **yukarıdan aşağıya SIRAYLA** çalıştır (her hücrede `Shift+Enter`).
3. Bir sonraki hücreye geçmeden önce, o hücrenin çıktısında **`✅`** işaretini gör.
   `✅` yoksa o adımı düzeltmeden ilerleme.

> Her adımın üstünde ne yaptığını, ne kadar süreceğini ve devam etmeden önce neyi
> kontrol etmen gerektiğini yazan mavi bir not var. Tahmin etmene gerek yok.


## 🔵 ADIM 1: GPU kontrolü
**Ne yapıyor:** Colab'ın sana ücretsiz bir GPU (Tesla T4) verip vermediğini kontrol eder.
**Ne kadar sürer:** ~5 saniye.
**Sonraki adıma geçmeden önce:** Çıktıda `✅ GPU HAZIR` yazısını gör. `⚠️` görürsen, notta yazan menü adımını yap ve bu hücreyi TEKRAR çalıştır.


In [ ]:
import subprocess

# nvidia-smi GPU'yu listeler; GPU yoksa hata verir.
proc = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(proc.stdout or proc.stderr)

if proc.returncode != 0 or "NVIDIA-SMI" not in (proc.stdout or ""):
    print("=" * 70)
    print("⚠️ GPU bulunamadı!")
    print("Üstteki menüden:  Runtime > Change runtime type >")
    print("                  Hardware accelerator > GPU (T4) seç,")
    print("sonra BU HÜCREYİ TEKRAR çalıştır.")
    print("=" * 70)
else:
    gpu_adi = "GPU"
    for satir in proc.stdout.splitlines():
        if "Tesla" in satir or "T4" in satir:
            gpu_adi = "Tesla T4" if "T4" in satir else satir.strip()
            break
    if "T4" not in proc.stdout:
        print("ℹ️ Not: Beklenen Tesla T4 değil ama bir GPU bulundu — devam edebilirsin.")
    print(f"✅ GPU HAZIR ({gpu_adi}) — ADIM 2'ye geç.")


## 🔵 ADIM 2: DiffDock + Remedia'yı kur
**Ne yapıyor:** DiffDock'un resmi kodunu ve senin Remedia repo'nu indirir, sonra gereken tüm Python paketlerini (torch_geometric, e3nn, fair-esm, rdkit vb.) kurar.
**Ne kadar sürer:** ~5–10 dakika (paket indirmesi yüzünden). Sabırlı ol, bir kere yapılır.
**Sonraki adıma geçmeden önce:** En sonda `✅ KURULUM TAMAM` yazısını gör. Hata görürsen README'deki "Sorun mu yaşıyorsun?" bölümüne bak.


In [ ]:
import os, sys, subprocess

# --- 1) Repoları klonla (zaten varsa tekrar klonlamaz) ---------------------
if not os.path.isdir("DiffDock"):
    subprocess.run(["git", "clone", "-q", "https://github.com/gcorso/DiffDock.git"], check=True)
if not os.path.isdir("Remedia"):
    subprocess.run(["git", "clone", "-q", "https://github.com/mehmetg06/Remedia.git"], check=True)
print("• Repolar klonlandı (DiffDock/, Remedia/)")

# --- 2) torch'a uygun PyG eklenti tekerleklerini (wheel) kur ---------------
import torch
TORCH = torch.__version__.split("+")[0]
CUDA = ("cu" + torch.version.cuda.replace(".", "")) if torch.version.cuda else "cpu"
PYG_URL = f"https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html"
print(f"• torch={TORCH}  cuda={CUDA}")

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# PyG çekirdeği + C++ eklentileri (DiffDock bunlara ihtiyaç duyar)
pip("torch_geometric")
pip("torch_scatter", "torch_sparse", "torch_cluster", "-f", PYG_URL)

# DiffDock'un geri kalan bağımlılıkları
pip("e3nn", "fair-esm", "spyrmsd", "prody", "biopython",
    "networkx", "pandas", "scipy", "pyyaml", "scikit-learn")

# rdkit (Colab'da olmayabilir)
try:
    import rdkit  # noqa: F401
except ImportError:
    pip("rdkit")

print("=" * 70)
print("✅ KURULUM TAMAM — ADIM 3'e geç.")
print("=" * 70)


## 🔵 ADIM 3: Girdi hazırlama — hangi molekülleri test edeceğim?
**Ne yapıyor:** Docklanacak reseptörü (protein) ve test edilecek molekülleri (ligand) hazırlar; sonra ekrana **kaç molekül** ve **hangi isimlerle** işleneceğini yazar.
**Ne kadar sürer:** ~10–30 saniye (reseptör indirmesi dahil).
**Sonraki adıma geçmeden önce:** Alttaki listede molekül isimlerini gözünle oku — **"evet, bunlar doğru moleküller"** dediğinde ADIM 4'e geç. Değilse aşağıdaki kutucukları düzelt ve hücreyi tekrar çalıştır.

> **Alanlar ne işe yarar?**
> - **UNIPROT_ID:** Reseptörün AlphaFold DB kimliği. Varsayılan `P30405` (CypD). Değiştirmezsen otomatik indirilir.
> - **RECEPTOR_PDB_PATH:** Elinde hazır bir `.pdb` varsa yolunu yaz (ör. `Remedia/data/hedef.pdb`). Boş bırakırsan UNIPROT_ID'den indirilir.
> - **SMILES_FILE:** Molekül listesinin `.smi` dosyası. Varsayılan repo örneği.
> - **CANDIDATES_CSV:** (opsiyonel) Sadece Vina'nın seçtiği en iyi adayları test etmek istersen `validated_candidates.csv` yolunu yaz. Boşsa `.smi` dosyasındaki TÜM moleküller test edilir.
> - **MAX_MOLECULES:** En fazla kaç molekül işlensin (T4'te zamanı kısa tutmak için).


In [ ]:
#@title ADIM 3 ayarları — kutucukları doldur, sonra bu hücreyi çalıştır { display-mode: "form" }
UNIPROT_ID = "P30405"  #@param {type:"string"}
RECEPTOR_PDB_PATH = ""  #@param {type:"string"}
SMILES_FILE = "Remedia/data/ligands_example.smi"  #@param {type:"string"}
CANDIDATES_CSV = ""  #@param {type:"string"}
MAX_MOLECULES = 5  #@param {type:"integer"}

import os, csv, urllib.request
from pathlib import Path

# --- 1) Reseptörü hazırla (verilen PDB yoksa AlphaFold'dan indir) ----------
if RECEPTOR_PDB_PATH.strip() and Path(RECEPTOR_PDB_PATH).exists():
    RECEPTOR_PDB = str(Path(RECEPTOR_PDB_PATH).resolve())
    print(f"• Reseptör (verilen dosya): {RECEPTOR_PDB}")
else:
    RECEPTOR_PDB = str(Path(f"receptor_{UNIPROT_ID}.pdb").resolve())
    if not Path(RECEPTOR_PDB).exists():
        url = f"https://alphafold.ebi.ac.uk/files/AF-{UNIPROT_ID}-F1-model_v4.pdb"
        print(f"• Reseptör indiriliyor: {url}")
        urllib.request.urlretrieve(url, RECEPTOR_PDB)
    print(f"• Reseptör (AlphaFold {UNIPROT_ID}): {RECEPTOR_PDB}")

# --- 2) SMILES dosyasını oku: {isim: smiles} -------------------------------
def parse_smi(path):
    mols = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 2:
                smiles, name = parts[0], parts[1]
            else:
                smiles, name = parts[0], f"mol_{len(mols)+1}"
            mols[name] = smiles
    return mols

if not Path(SMILES_FILE).exists():
    raise FileNotFoundError(
        f"SMILES dosyası bulunamadı: {SMILES_FILE}\n"
        f"Doğru yolu yaz (ör. Remedia/data/ligands_example.smi) ve tekrar çalıştır."
    )
smiles_map = parse_smi(SMILES_FILE)
print(f"• SMILES dosyasında {len(smiles_map)} molekül var: {SMILES_FILE}")

# --- 3) (opsiyonel) Vina adaylarına göre filtrele/sırala -------------------
secilen_isimler = list(smiles_map.keys())
if CANDIDATES_CSV.strip() and Path(CANDIDATES_CSV).exists():
    aday_isimler = []
    with open(CANDIDATES_CSV, newline="") as f:
        for row in csv.DictReader(f):
            ad = str(row.get("ligand", "")).strip()
            if ad and ad != "(yok)":
                aday_isimler.append(ad)
    eslesmeyen = [a for a in aday_isimler if a not in smiles_map]
    secilen_isimler = [a for a in aday_isimler if a in smiles_map]
    print(f"• CANDIDATES_CSV kullanıldı: {len(secilen_isimler)} aday SMILES ile eşleşti.")
    if eslesmeyen:
        print(f"  ⚠️ SMILES bulunamayan (atlanan) adaylar: {', '.join(eslesmeyen)}")
elif CANDIDATES_CSV.strip():
    print(f"  ⚠️ CANDIDATES_CSV bulunamadı ({CANDIDATES_CSV}); .smi dosyasındaki tüm moleküller kullanılacak.")

# --- 4) MAX_MOLECULES ile sınırla ve listeyi kur ---------------------------
secilen_isimler = secilen_isimler[:MAX_MOLECULES]
molecules = [(ad, smiles_map[ad]) for ad in secilen_isimler]

# --- 5) Kullanıcı görsel onayı için TABLOYU YAZDIR -------------------------
print()
print("=" * 70)
print(f"İŞLENECEK MOLEKÜLLER  ({len(molecules)} adet)")
print("-" * 70)
print(f"{'#':>2}  {'İSİM':<24} SMILES")
print("-" * 70)
for i, (name, smiles) in enumerate(molecules, 1):
    print(f"{i:>2}  {name:<24} {smiles}")
print("=" * 70)
if not molecules:
    print("⚠️ Hiç molekül seçilmedi! SMILES_FILE / CANDIDATES_CSV ayarlarını kontrol et.")
else:
    print("👀 Yukarıdaki isimler DOĞRU moleküllerse ADIM 4'e geç.")


## 🔵 ADIM 4: DiffDock'u çalıştır
**Ne yapıyor:** Her molekülü tek tek DiffDock'a verir; protein üzerinde en olası bağlanma pozunu ve bir **güven skoru** üretir.
**Ne kadar sürer:** Molekül başına ~1–3 dakika (T4'te). İlk molekülde ESM protein modeli indirileceği için biraz daha uzun sürebilir.
**Sonraki adıma geçmeden önce:** İlerleme çubuğu dolup her molekül için `✅ ... güven skoru: X` satırını gör. Bir molekül `❌` verirse sorun değil — diğerleri devam eder, o satır boş güvenle raporlanır.


In [ ]:
import re, sys, subprocess
from pathlib import Path
from tqdm.auto import tqdm

OUT_ROOT = Path("diffdock_out")
OUT_ROOT.mkdir(exist_ok=True)

results_rows = []  # {ligand, confidence, pose_path}

print(f"Toplam {len(molecules)} molekül işlenecek.\n")

for name, smiles in tqdm(molecules, desc="DiffDock", unit="mol"):
    out_dir = (OUT_ROOT / name).resolve()
    in_csv = (OUT_ROOT / f"{name}_input.csv").resolve()
    with open(in_csv, "w") as f:
        f.write("complex_name,protein_path,ligand_description,protein_sequence\n")
        f.write(f"{name},{RECEPTOR_PDB},{smiles},\n")

    cmd = [sys.executable, "-m", "inference",
           "--config", "default_inference_args.yaml",
           "--protein_ligand_csv", str(in_csv),
           "--out_dir", str(out_dir)]
    proc = subprocess.run(cmd, cwd="DiffDock", capture_output=True, text=True)

    # En iyi pozu (rank1) ve güven skorunu çıktı dosyasından bul
    conf, pose_path = None, ""
    sdfs = sorted(out_dir.glob("**/rank1_confidence*.sdf"))
    if not sdfs:
        sdfs = sorted(out_dir.glob("**/rank1*.sdf"))
    if sdfs:
        pose_path = str(sdfs[0])
        m = re.search(r"confidence(-?[\d.]+)\.sdf", sdfs[0].name)
        if m:
            conf = float(m.group(1))

    if pose_path:
        skor_txt = f"{conf:.3f}" if conf is not None else "(dosyada skor yok)"
        print(f"✅ {name} tamamlandı, güven skoru: {skor_txt}")
    else:
        # Gerçek hatayı göster (son birkaç satır) — kullanıcı sebebi görsün
        hata = (proc.stderr or proc.stdout or "").strip().splitlines()
        son = " | ".join(hata[-3:]) if hata else "bilinmeyen hata"
        print(f"❌ {name} başarısız — sebep: {son}")

    results_rows.append({"ligand": name, "confidence": conf, "pose_path": pose_path})

basarili = sum(1 for r in results_rows if r["pose_path"])
print(f"\n✅ DiffDock bitti: {basarili}/{len(results_rows)} molekül başarıyla docklandı. ADIM 5'e geç.")


## 🔵 ADIM 5: Sonuçları indir
**Ne yapıyor:** Tüm skorları tek bir `diffdock_results.csv` dosyasına yazar ve **otomatik olarak bilgisayarına indirir** — elle bir şey aramana gerek yok.
**Ne kadar sürer:** ~5 saniye.
**Sonraki adıma geçmeden önce:** Tarayıcın `diffdock_results.csv` dosyasını indirdiğinde işin bitti. Alttaki büyük yeşil mesajdaki talimatı izle.


In [ ]:
import csv
from google.colab import files

res_csv = "diffdock_results.csv"
with open(res_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["ligand", "diffdock_confidence", "diffdock_pose_path"])
    for r in results_rows:
        writer.writerow([
            r["ligand"],
            "" if r["confidence"] is None else round(r["confidence"], 4),
            r["pose_path"],
        ])

print(f"'{res_csv}' yazıldı ({len(results_rows)} satır). İndiriliyor...")
files.download(res_csv)  # tarayıcı otomatik indirir

print()
print("#" * 72)
print("#" + " " * 70 + "#")
print("#" + "🎉  BİTTİ!".center(69) + "#")
print("#" + " " * 70 + "#")
print("#" + "İndirilen  diffdock_results.csv  dosyasını".center(70) + "#")
print("#" + "Codespaces'teki  Remedia/results/  klasörüne yükle,".center(70) + "#")
print("#" + "sonra Codespaces terminalinde ŞUNU çalıştır:".center(70) + "#")
print("#" + " " * 70 + "#")
print("#" + "python src/merge_diffdock_results.py".center(70) + "#")
print("#" + " " * 70 + "#")
print("#" * 72)
